# Chapter 5: Linear Models and Regularization

## Ringkasan Bab & Penjelasan Teoretis
Bab ini mengupas model regresi linear ($y = \beta_0 + \beta_1 x_1 + \dots + \beta_n x_n + \epsilon$) serta teknik **regularisasi** untuk mengatasi multikolinearitas dan mencegah *overfitting* melalui pengendalian *bias-variance trade-off*.

### 1. Regresi Linear, Ridge ($L_2$), Lasso ($L_1$), dan ElasticNet
* **Ridge Regression ($L_2$)**: Menambahkan penalti kuadrat magnitudo koefisien ($\alpha \sum_{j=1}^{p} \beta_j^2$) ke dalam fungsi kerugian OLS. Menyusutkan koefisien mendekati nol tanpa menghapus fitur.
* **Lasso Regression ($L_1$)**: Menambahkan penalti nilai absolut koefisien ($\alpha \sum_{j=1}^{p} |\beta_j|$). Mampu mendorong koefisien fitur yang tidak relevan menjadi **tepat nol** (*sparsity* / seleksi fitur otomatis).
* **ElasticNet**: Menggabungkan penalti $L_1$ dan $L_2$ yang dikontrol melalui parameter `alpha` dan `l1_ratio`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

X, y = make_regression(n_samples=1000, n_features=100, n_informative=10, noise=20, random_state=123)

np.random.seed(123)
for i in range(50, 100):
    X[:, i] = X[:, i-50] + np.random.normal(0, 0.1, size=1000)

feature_names = [f'feature_{i}' for i in range(100)]
X_plot = X[:, 0].reshape(-1, 1)

y = y * 1000
y = y + np.sin(X_plot.ravel()) * 150 + np.exp(X_plot.ravel()/10)

df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

linear_model = LinearRegression().fit(X_train, y_train)
ridge_model = Ridge(alpha=1.0).fit(X_train, y_train)
lasso_model = Lasso(alpha=10.0, max_iter=10000, tol=0.001).fit(X_train, y_train)
elastic_model = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=123, max_iter=10000, tol=1e-4).fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)
y_pred_ridge = ridge_model.predict(X_test)
y_pred_lasso = lasso_model.predict(X_test)
y_pred_elastic = elastic_model.predict(X_test)

metrics_df = pd.DataFrame({
    'Model': ['ElasticNet Regression', 'Ridge Regression', 'Lasso Regression', 'Linear Regression'],
    'Mean Squared Error': [
        mean_squared_error(y_test, y_pred_elastic),
        mean_squared_error(y_test, y_pred_ridge),
        mean_squared_error(y_test, y_pred_lasso),
        mean_squared_error(y_test, y_pred_linear)
    ],
    'R-squared': [
        r2_score(y_test, y_pred_elastic),
        r2_score(y_test, y_pred_ridge),
        r2_score(y_test, y_pred_lasso),
        r2_score(y_test, y_pred_linear)
    ]
}).sort_values('Mean Squared Error', ascending=True)

display(metrics_df)

# Visualisasi Perbandingan Garis Regresi
plt.figure(figsize=(10, 6))
plt.scatter(X_test[:, 0], y_test, color='blue', alpha=0.5, label='Test Data')
X_line = np.linspace(df['feature_0'].min(), df['feature_0'].max(), 100).reshape(-1, 1)
X_line_full = np.zeros((100, len(feature_names)))
X_line_full[:, 0] = X_line.ravel()

plt.plot(X_line, ridge_model.predict(X_line_full), color='red', label='Ridge Regression')
plt.plot(X_line, lasso_model.predict(X_line_full), color='green', linestyle='--', label='Lasso Regression')
plt.plot(X_line, linear_model.predict(X_line_full), color='purple', label='Linear Regression')
plt.xlabel('feature_0')
plt.ylabel('target')
plt.title('Ridge, Lasso, and Linear Regression Comparison')
plt.legend()
plt.show()

# ElasticNet Coefficient Paths
alphas = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
plt.figure(figsize=(10, 6))
for l1_ratio in l1_ratios:
    coefs = []
    for alpha in alphas:
        em = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=123, max_iter=10000, tol=1e-4).fit(X_train, y_train)
        coefs.append(em.coef_)
    for feature_idx in range(X_train.shape[1]):
        plt.plot(
            np.log10(alphas), np.array(coefs)[:, feature_idx],
            label=f'l1_ratio={l1_ratio}, feature_{feature_idx}' if feature_idx == 0 else "",
            alpha=0.5, linestyle=['solid', 'dashed', 'dotted', 'dashdot', '-'][l1_ratios.index(l1_ratio)]
        )
plt.xlabel('log(alpha)')
plt.ylabel('Coefficient value')
plt.title('ElasticNet Coefficient Paths')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

### 2. Polynomial Regression & Spline Interpolation
* **Polynomial Regression**: Memetakan fitur input ke derajat polinomial ($x, x^2, x^3, \dots$) untuk memodelkan tren kurvilinear non-linear.
* **Spline Interpolation (`SplineTransformer`)**: Membagi domain fitur pada titik-titik **knots** dan mencocokkan polinomial sepotong-sepotong (*piecewise polynomials*) yang mulus dan kontinu.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.pipeline import make_pipeline

np.random.seed(123)
n_samples = 1000
X_nl = np.random.uniform(-50, 50, (n_samples, 1))
y_nl = 2 * X_nl[:, 0] + 27 * np.sin(X_nl[:, 0] / 8) + np.random.normal(0, 4, n_samples)
data = pd.DataFrame(X_nl, columns=['feature1'])
data['target'] = y_nl

X_sub = data[['feature1']]
y_sub = data['target']
degrees = [1, 2, 3, 4, 5]
results = []

for degree in degrees:
    poly = PolynomialFeatures(degree=degree)
    X_poly = poly.fit_transform(X_sub)
    X_tr, X_te, y_tr, y_te = train_test_split(X_poly, y_sub, test_size=0.2, random_state=123)
    poly_model = LinearRegression().fit(X_tr, y_tr)
    y_p = poly_model.predict(X_te)
    results.append({'Degree': degree, 'MSE': mean_squared_error(y_te, y_p), 'R2': r2_score(y_te, y_p)})

results_df = pd.DataFrame(results)
display(results_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
ax1.scatter(X_sub['feature1'], y_sub, alpha=0.2, label='Data points')
X_plot = np.linspace(X_sub['feature1'].min(), X_sub['feature1'].max(), 1000).reshape(-1, 1)
colors = ['blue', 'green', 'red', 'purple', 'black']

for degree, color in zip(degrees, colors):
    poly = PolynomialFeatures(degree=degree)
    model = LinearRegression().fit(poly.fit_transform(X_sub), y_sub)
    ax1.plot(X_plot, model.predict(poly.transform(X_plot)), label=f'Degree {degree}', color=color, linewidth=2)
ax1.set_title('Polynomial Regression Fits')
ax1.legend()

# Spline Interpolation
n_knots = [3, 5, 7]
ax2.scatter(X_sub, y_sub, color='black', alpha=0.3, label='Data points')
for n_knot, color in zip(n_knots, colors[:len(n_knots)]):
    spline_model = make_pipeline(SplineTransformer(n_knots=n_knot, degree=3), LinearRegression()).fit(X_sub, y_sub)
    ax2.plot(X_plot, spline_model.predict(X_plot), label=f'Spline (knots={n_knot})', color=color, linewidth=2)
ax2.set_title('Spline Interpolation with Different Numbers of Knots')
ax2.legend()
plt.tight_layout()
plt.show()